In [ ]:
from logging import getLogger, FATAL
import os
import numpy as np
from caret_analyze import Architecture, Application, check_procedure, Lttng, LttngEventFilter
from caret_analyze.plot import Plot, chain_latency
from bokeh.plotting import reset_output, output_notebook, figure, show
reset_output()
output_notebook()
import bokeh
print(f"Python Version: {bokeh.__version__}")

logger = getLogger()
logger.setLevel(FATAL)

In [ ]:
tracing_log_path = [
    '<path/to/caret_trace_data>'
]


In [ ]:
arch = Architecture('lttng', tracing_log_path)
lttng = Lttng(tracing_log_path)

In [ ]:
target_path_json_list = [
    '/sensor_simulator_node',
    '/bridge_to_agnocast_node',
    '/relay_by_pattern_a_node',
    '/relay_by_pattern_b_node',
    '/take_by_pattern_b_node',
    '/bridge_from_agnocast_node',
    '/dummy_actuator_node',
]

In [ ]:
architecture_raw = 'architecture_raw.yaml'
architecture_path = 'architecture_path.yaml'
architecture_path_fixed = 'architecture_path_fixed.yaml'
arch = Architecture('lttng', tracing_log_path)
arch.export(architecture_raw, force=True)

In [ ]:
def update_arch():
    # Search path
    arch = Architecture('yaml', architecture_raw)

    target_path_list = arch.search_paths(
        *target_path_json_list, max_node_depth=5)
    print(target_path_list)
    arch.add_path('target_path', target_path_list[0])

    # update message_context
    for path in arch.paths:
        for n in path.node_paths:
            if n.publish_topic_name is None or n.subscribe_topic_name is None:
                continue
            if not n.message_context:
                arch.update_message_context(
                    n.node_name,
                    'use_latest_message',
                    n.subscribe_topic_name,
                    n.publish_topic_name,
                )

    arch.export(architecture_path, force=True)
    for i, path in enumerate(target_path_list):
        print(f"--- Path {i} ---")
        for node_path in path.node_paths:
            print(node_path.node_name)


update_arch()

In [ ]:
arch = Architecture('yaml', architecture_path)
app = Application(arch, lttng)

target_paths = app.paths

In [ ]:
import time

class Timer:
    def __init__(self, format_str="{:.3f}[s]"):
        self.format_str = format_str
        self.start = None
        self.end = None

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time.time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.time()
        out_str = self.format_str.format(self.duration)
        self.start = None
        self.end = None
        print(out_str)


## create_message_flow_plot(Path)

In [ ]:
for target_path in target_paths:
    for granularity in ['raw', 'node']:
        with Timer():
            Plot.create_message_flow_plot(
                target_path, granularity=granularity).show()


## create_response_time_stacked_bar_plot

In [ ]:
for target_path in target_paths:
    for case in ['all', 'best', 'worst', 'worst-with-external-latency']:
        with Timer():
            Plot.create_response_time_stacked_bar_plot(
                target_path, case=case).show()


## create_period_timeseries_plot(callbacks|communications|publishers|subscriptions)

In [ ]:
for target_path in target_paths:
    for obj in [app.callbacks, app.communications, app.publishers, app.subscriptions]:
        with Timer():
            Plot.create_period_timeseries_plot(obj).show()


## create_frequency_timeseries_plot(callbacks|communications|publishers|subscriptions)

In [ ]:
for target_path in target_paths:
    for obj in [app.callbacks, app.communications, app.publishers, app.subscriptions]:
        with Timer():
            Plot.create_frequency_timeseries_plot(obj).show()


## create_latency_timeseries_plot(callbacks|communications)

In [ ]:
for target_path in target_paths:
    for obj in [app.callbacks, app.communications]:
        with Timer():
            Plot.create_latency_timeseries_plot(obj).show()


## create_response_time_timeseries_plot(Path)

In [ ]:
for target_path in target_paths:
    for case in ['all', 'best', 'worst', 'worst-with-external-latency']:
        with Timer():
            Plot.create_response_time_timeseries_plot(
                target_path, case=case).show()

for target_path in target_paths:
    for case in ['all', 'best', 'worst', 'worst-with-external-latency']:
        with Timer():
            Plot.create_response_time_timeseries_plot(
                app.paths, case=case).show()


## create_frequency_histogram_plot(callbacks|communications|publishers|subscriptions)

In [ ]:
for target_path in target_paths:
    for obj in [app.callbacks, app.communications, app.publishers, app.subscriptions]:
        with Timer():
            Plot.create_frequency_histogram_plot(obj).show()


## create_latency_histogram_plot(callbacks|communications)

In [ ]:
for target_path in target_paths:
    for obj in [app.callbacks, app.communications]:
        with Timer():
            Plot.create_latency_histogram_plot(obj).show()


## create_period_histogram_plot(callbacks|communications|publishers|subscriptions)

In [ ]:
for target_path in target_paths:
    for obj in [app.callbacks, app.communications, app.publishers, app.subscriptions]:
        with Timer():
            Plot.create_period_histogram_plot(obj).show()


## create_response_time_histogram_plot(Path)

In [ ]:
for target_path in target_paths:
    for case in ['all', 'best', 'worst', 'worst-with-external-latency']:
        with Timer():
            Plot.create_response_time_histogram_plot(
                app.paths, case=case).show()

for target_path in target_paths:
    for case in ['all', 'best', 'worst', 'worst-with-external-latency']:
        with Timer():
            Plot.create_response_time_histogram_plot(
                app.paths, case=case).show()


## create_callback_scheduling_plot(node)

In [ ]:
from caret_analyze.exceptions import ItemNotFoundError

try:
    #target_node = app.get_node(node_name)
    
    #objs = [app, *app.executors, *app.paths, target_node, *app.callback_groups]
    objs = [app, *app.paths]
    
    for obj in objs:
        try:
            with Timer():
                Plot.create_callback_scheduling_plot(obj).show()
        except ItemNotFoundError:
            print(f"Skipping empty object in {node_name}")
except ItemNotFoundError:
    print(f"Node {node_name} not found. Skipping...")
            

## chain_latency(Path)

In [ ]:
for target_path in target_paths:
    for granularity in ['node', 'end-to-end']:
        dot = chain_latency(target_path, granularity='node',
                            lstrip_s=1, rstrip_s=1)  # time_id='system_time'
        display(dot)
